In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

In [2]:
#Uncomment and run if needed

#dbutils.fs.rm("dbfs:/FileStore/input/wrongPath", True)





# Data Cleansing & Anonymization

We will be using Spark to load data from a dataset about people in NSS

We are going to clean data from a file that contains data about different employees, with several columns, among them:

* first name
* middle name
* last name
* ssn (Social security number)
* birthdate
* salary
* company
* department
* role


There are several issues:

* Name format: first name may not be right, for instance we may have Rachel or CAROL
* SSN format: Some of the records come without hyphens. for instance  628-40-9043 and 324469083
* Due to the format issues, data come with duplicates, your mission is to fix it.

Once the issues are fixed, records are guaranteed to match, so you can remove duplicates easily


* Also you should anonymize SSN 



In [3]:
from pyspark.sql.functions import split, col

file_path = "/home/jovyan/work/datasets/employees/employees.csv"



raw_df = spark.read.csv(file_path,
                         header="true", 
                          inferSchema="true")

raw_df.count()


100160

In [4]:
from pyspark.sql.functions import concat, substring, lit

# A function that fix SSN to proper format XXX-XX-XXXX
def format_ssn(ssn):
    return concat(substring(ssn, 1, 3), lit('-'), substring(ssn, 4, 2), lit('-'), substring(ssn, 6, 4))

# Filter Columns not containing ssn
# Create a new column with the same name fixing the ssn column
wrong_ssn_df = raw_df.filter(~col("ssn").contains("-")) \
.withColumn("ssn", format_ssn(col("ssn"))) 

# Filter again the original dataframe to keep only the rows containing a - (good rows)
# Union it with the dataset with the fixed SSN
fixed_ssn_df = raw_df.filter(col("ssn").contains("-"))\
.union(wrong_ssn_df)


In [5]:
from pyspark.sql.functions import  initcap

#Use initcap function, that giving a String returns it with inly the first letter in uppercase
#No need to bifurcate dataset this time since good rows are not afected by the fix, they stay the same
fixed_name_df = fixed_ssn_df \
.withColumn("first_name", initcap(col("first_name")))


In [6]:
#Now that all the rows are clear, we can safely remove duplicates
fixed_df = fixed_name_df.drop_duplicates()

fixed_df.show(truncate=False)


+-----------+-----------+----------+-----------+----------+----------------+---------+---------------------------+-----------+--------------------+
|first_name |middle_name|last_name |ssn        |birthdate |birthplace      |salary   |company                    |department |role                |
+-----------+-----------+----------+-----------+----------+----------------+---------+---------------------------+-----------+--------------------+
|Maurice    |Jennifer   |Ellis     |064-29-7666|1956-02-06|Lake Amber      |67029.31 |Hahn-Walker                |Support    |Customer Support    |
|Amy        |Jason      |Meyers    |085-34-2673|1976-02-03|Emilyburgh      |146274.13|Armstrong Inc              |Product    |UX Designer         |
|Michelle   |Amanda     |Ritter    |284-60-0722|1961-01-22|Jasonfurt       |97073.14 |Smith, Moore and Fields    |Support    |Support Specialist  |
|Kimberly   |Gregory    |Calderon  |236-32-7600|1958-12-13|Port Hannahside |96551.07 |Lopez-Becker              

In [7]:
from pyspark.sql.functions import sha2

#Now we want to anonymize the salary and the ssn
#Use sha2 function with the column and the number of bits(256) inside withColumn
#Beware that it only applies on String, you need to cast salary to string first (or at the time)
anon_df = fixed_df \
  .withColumn("salary", sha2(col("salary").cast("string"), 256)) \
  .withColumn("ssn", sha2(col("ssn"), 256))


anon_df.show(truncate=False)


+-----------+-----------+----------+----------------------------------------------------------------+----------+----------------+----------------------------------------------------------------+---------------------------+-----------+--------------------+
|first_name |middle_name|last_name |ssn                                                             |birthdate |birthplace      |salary                                                          |company                    |department |role                |
+-----------+-----------+----------+----------------------------------------------------------------+----------+----------------+----------------------------------------------------------------+---------------------------+-----------+--------------------+
|Maurice    |Jennifer   |Ellis     |a7fd69f521a97c510f122d106240bf2ee399039c0383397aed898801d65b1da8|1956-02-06|Lake Amber      |faee4d2e82250414fb6a91fdda7e627985bc5d65448281993623d5dfe2c828fe|Hahn-Walker                |Support   